# Induction Circuit Analysis

Deep analysis of induction circuits: full path tracing through
embedding → previous-token head → induction head, matching score
matrices, copy circuit verification, prefix search behavior, and
circuit completeness testing.

References:
- Olsson et al. (2022) "In-context Learning and Induction Heads"
- Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"

## Why This Matters

Circuit induction helps identify and characterize the specific subnetworks responsible for model behaviors. Finding circuits — the algorithms models actually implement — is the core goal of mechanistic interpretability.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.induction_circuit_analysis import (
    induction_circuit_path_tracing,
    matching_score_matrix,
    copy_circuit_verification,
    prefix_search_analysis,
    induction_circuit_completeness,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
# Repeated pattern for induction
tokens = jnp.array([1, 2, 3, 4, 5, 1, 2, 3, 4, 5])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print("Model ready")

## Induction Circuit Path Tracing

Identify previous-token heads and induction heads, then trace
how information flows through the full circuit.

In [ ]:
result = induction_circuit_path_tracing(model, tokens)
print(f"Best circuit: L{result['best_circuit'][0]}H{result['best_circuit'][1]} -> L{result['best_circuit'][2]}H{result['best_circuit'][3]}")
print(f"  Strength: {result['best_circuit'][4]:.4f}")
print(f"\nPrev-token scores:")
for l in range(cfg.n_layers):
    scores = [f"H{h}:{result['prev_token_scores'][l,h]:.3f}" for h in range(cfg.n_heads)]
    print(f"  L{l}: {', '.join(scores)}")
print(f"\nInduction scores:")
for l in range(cfg.n_layers):
    scores = [f"H{h}:{result['induction_scores'][l,h]:.3f}" for h in range(cfg.n_heads)]
    print(f"  L{l}: {', '.join(scores)}")

## Matching Score Matrix

In [ ]:
result = matching_score_matrix(model, tokens)
print("Best matching heads:")
for l, h, s in result['best_matching_heads'][:5]:
    print(f"  L{l}H{h}: {s:.4f}")
print(f"\nMean match by layer:")
for l in range(cfg.n_layers):
    print(f"  Layer {l}: {result['mean_match_by_layer'][l]:.4f}")

## Copy Circuit Verification

Check if a head implements a copy circuit by projecting its
OV output through the unembedding.

In [ ]:
for l in range(min(2, cfg.n_layers)):
    for h in range(min(2, cfg.n_heads)):
        result = copy_circuit_verification(model, tokens, layer=l, head=h)
        print(f"L{l}H{h}: copy_score={result['copy_score']:.4f}, "
              f"accuracy={result['token_copy_accuracy']:.1%}, "
              f"ov_trace={result['ov_trace']:.4f}")

## Prefix Search Analysis

In [ ]:
result = prefix_search_analysis(model, tokens)
print("Best prefix-matching heads:")
for l, h, s in result['best_prefix_heads'][:5]:
    print(f"  L{l}H{h}: {s:.4f}")

## Circuit Completeness

In [ ]:
result = induction_circuit_completeness(model, tokens, metric_fn)
print(f"Base metric: {result['base_metric']:.4f}")
print(f"Circuit faithfulness: {result['circuit_faithfulness']:.4f}")
print(f"Redundancy score: {result['redundancy_score']:.4f}")
print(f"\nCircuit heads ({len(result['circuit_heads'])}):")
for role, l, h in result['circuit_heads'][:10]:
    effect = result['ablation_effects'].get((role, l, h), 0)
    print(f"  {role} L{l}H{h}: effect={effect:+.4f}")